# 04 - Importance Measurement

Plan section 39: the importance system produces real, device-safe, restoring signals and fossils.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Importance-system measurement (plan section 39)

Validates that the measurable signals exist and behave: per-expert
routing load, gradient norm, parameter norm, and temporal stability
extracted after a real backward pass, a temporary-ablation loss
delta, and the append-only lifecycle fossil ledger the studies
serialize.


In [ ]:
import torch
from act.model import StaticMoEConfig, StaticMoECausalLM
from act.importance import (
    LifecycleLedger,
    expert_ablation_delta,
    measure_expert_signals,
)

model = StaticMoECausalLM(StaticMoEConfig(
    vocab_size=32, context_size=12, hidden_size=24, num_layers=2,
    num_heads=4, ffn_size=24, num_experts=4, active_experts=2,
))
tokens = torch.randint(0, 32, (2, 12))
loss = model.loss(tokens)
loss.backward()  # gradient signals exist only after a real step
print('training loss:', round(float(loss.detach()), 4))
signals = measure_expert_signals(model)
assert len(signals) == 8  # 2 layers x 4 experts
first = signals[0]
print(f"{first.layer_id}/{first.expert_id}: load={first.routing_load:.3f} "
      f"grad={first.gradient_norm:.4f} param={first.parameter_norm:.3f}")

In [ ]:
# Temporary ablation: zeroing one expert moves the loss by a measured
# delta (positive or negative at toy scale - an unused expert can be
# harmless) and the parameters must be restored afterwards.
baseline = float(model.loss(tokens).detach())
delta = expert_ablation_delta(model, tokens, 'L0E0')
restored = float(model.loss(tokens).detach())
assert restored == baseline, 'ablation did not restore parameters'
print('ablation delta for L0E0:', round(delta, 6),
      '| baseline restored:', restored == baseline)

In [ ]:
# Lifecycle fossils: the append-only record of every accepted mutation.
from act.topology import create_genome, genome_hash, grow_expert

genome = create_genome(42, hidden_size=24, vocab_size=32, depth=1,
                       ffn='moe', expert_count=4, active_experts=2)
hash_before = genome_hash(genome)
added = grow_expert(genome, genome.layers[0].id, step=12)
assert added is not None
ledger = LifecycleLedger()
fossil = ledger.record(
    step=12, operation='grow', expert_id=added, layer_id='L0',
    architecture_hash_before=hash_before,
    architecture_hash_after=genome_hash(genome),
    signal=first,
)
assert fossil.signal['expert_id'] == first.expert_id
print('fossil:', fossil.operation, fossil.expert_id,
      '| ledger lines:', len(ledger.to_jsonl().splitlines()))

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '04_importance',
 'signals': len(signals),
 'ablation_delta': delta,
 'ablation_restored': restored == baseline,
 'ablation_finite': delta == delta and abs(delta) != float('inf'),
 'fossils': len(ledger.records)}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '04-importance.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)